# CatBoost y Naive Bayes en inferencia multilingüe

**Bloque 1 · Javi y Aitor.** Estudiamos el efecto de la lematización y de representar explícitamente la diferencia entre premisa e hipótesis. Usamos el texto en su lengua original. La prueba de traducción será otra comparación sobre los mismos IDs. Los CSV oficiales están en `data/`. [Datos de la competición](https://www.kaggle.com/competitions/contradictory-my-dear-watson/data) · [Protocolo y reparto](JAVI_AITOR.md)

**Registro histórico.** Estas celdas muestran las métricas guardadas. Para ejecutar experimentos nuevos, usa [el cuaderno de trabajo](workflow_javi_aitor.ipynb).

In [1]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
PROJECT = cwd if (cwd / "run_experiment_matrix.py").is_file() else cwd / "0-AiDea"
if not (PROJECT / "data/train.csv").is_file():
    raise RuntimeError("Abre el kernel desde training o desde 0-AiDea.")
sys.path.insert(0, str(PROJECT))

import pandas as pd
from FINAL.translator import load_translations
from training import LABEL_NAMES

def read_csv(path):
    return pd.read_csv(PROJECT / path, dtype={"id": str}, keep_default_na=False)

source = read_csv("data/train.csv")
train = read_csv("artifacts/splits/train.csv")
val = read_csv("artifacts/splits/val.csv")
all_rows = pd.concat([train, val]).set_index("id").sort_index()
assert all_rows.index.is_unique
pd.testing.assert_frame_equal(all_rows, source.set_index("id").sort_index())
translation_path = PROJECT / "artifacts/translations/train_gemma3_4b.csv"
translations = load_translations(source, translation_path).set_index("id") if translation_path.is_file() else None
print(len(train), "pares de desarrollo y", len(val), "de validación.")

C:\Users\javiy\Documents\training\0-AiDea\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pandas 3.1.1
scikit-learn 1.9.1
spacy 3.8.18
en-core-web-sm 3.8.0
catboost 1.2.10
9696 pares de desarrollo y 2424 de validación.


## Selección en desarrollo

Agrupamos por premisa original en tres folds con semilla 42. Probamos dos preprocesamientos con los mismos ratios POS y características de relación. Comparar `pair_diff` con `pair_separate` mide el efecto conjunto de añadir diferencia TF-IDF y solapamiento. El vocabulario y las 300 características seleccionadas con chi-cuadrado se ajustan en cada fold. Multinomial Naive Bayes combina evidencias léxicas bajo independencia condicional. CatBoost puede aprender interacciones mediante árboles sucesivos que reducen la pérdida multiclase. Fijamos sus hiperparámetros para limitar el número de comparaciones.

In [2]:
archive = PROJECT / "results_javi_aitor_cv"
records = json.loads((archive / "results.json").read_text(encoding="utf-8"))["experiments"]
assert len(records) == 8
table = pd.DataFrame([{"dataset": r["dataset"], "representación": r["representation"],
                       "modelo": r["classifier"], **r["summary"]} for r in records])
table[["dataset", "representación", "modelo", "accuracy_mean", "macro_f1_mean",
       "fit_seconds_total"]].sort_values("accuracy_mean", ascending=False).round(4)

,dataset,representación,modelo,accuracy_mean,macro_f1_mean,fit_seconds_total
1,baseline,pair_diff,catboost,0.5219,0.5208,8.1
5,without_reduction,pair_diff,catboost,0.5182,0.5171,8.2
0,baseline,pair_diff,naive_bayes,0.4850,0.4828,1.9
4,without_reduction,pair_diff,naive_bayes,0.4829,0.4807,1.7
3,baseline,pair_separate,catboost,0.4766,0.4702,6.4
7,without_reduction,pair_separate,catboost,0.4727,0.4664,7.2
6,without_reduction,pair_separate,naive_bayes,0.4426,0.4426,1.5
2,baseline,pair_separate,naive_bayes,0.4425,0.4424,1.5


## Validación de los dos finalistas

Elegimos una configuración por algoritmo usando la accuracy media de desarrollo y macro-F1 para desempatar. Ambos finalistas predicen los mismos 2424 pares de validación. El split histórico comparte premisas entre desarrollo y validación, de modo que las medidas de generalización a premisas nuevas proceden de los folds agrupados. Mostramos tiempos de ajuste y evaluación del modelo sobre datos ya preprocesados.

In [3]:
cv_archive = PROJECT / "results_translation_cv"
cv_records = json.loads((cv_archive / "results.json").read_text(encoding="utf-8"))["experiments"]
nb_holdout = next(r["holdout"] for r in cv_records
                  if (r["dataset"], r["classifier"]) == ("original", "naive_bayes"))
cb_holdout = json.loads((PROJECT / "results_ollama_catboost/metrics.json").read_text(encoding="utf-8"))["arms"]["original"]
nb_csv = cv_archive / nb_holdout["predictions_path"].replace("\\", "/").split("/")[-1]
nb_predictions = read_csv(nb_csv).set_index("id").loc[val.id, "prediction"].to_numpy()
cb_predictions = read_csv("results_ollama_catboost/predictions.csv").set_index("id").loc[val.id, "original"].to_numpy()

models = {}
for name, holdout, predictions in (("naive_bayes", nb_holdout, nb_predictions),
                                   ("catboost", cb_holdout, cb_predictions)):
    best = max((r for r in records if r["classifier"] == name),
               key=lambda r: (r["summary"]["accuracy_mean"], r["summary"]["macro_f1_mean"]))
    metrics = {**holdout, "per_language": pd.DataFrame(holdout["per_language"])}
    models[name] = (best, metrics, predictions)
pd.DataFrame([{"modelo": name, "dataset": r["dataset"], "representación": r["representation"],
               "accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "segundos": m["runtime_seconds"]}
              for name, (r, m, _) in models.items()]).round(4)

,modelo,dataset,representación,accuracy,macro_f1,segundos
0,naive_bayes,baseline,pair_diff,0.4847,0.4828,1.7
1,catboost,baseline,pair_diff,0.5206,0.5195,5.6


In [4]:
for name, (_, metrics, _) in models.items():
    print(name, "· filas = clase real, columnas = predicción")
    display(pd.DataFrame(metrics["confusion_matrix"], index=LABEL_NAMES.values(), columns=LABEL_NAMES.values()))
    display(pd.DataFrame(metrics["classification_report"]).T.loc[list(LABEL_NAMES.values())].round(3))
    display(metrics["per_language"].sort_values("n", ascending=False).round(3))

naive_bayes · filas = clase real, columnas = predicción


,entailment,neutral,contradiction
entailment,468,197,170
neutral,268,354,154
contradiction,238,222,353


,precision,recall,f1-score,support
entailment,0.480,0.560,0.517,835.0
neutral,0.458,0.456,0.457,776.0
contradiction,0.521,0.434,0.474,813.0


,language,n,accuracy,macro_f1,n_classes
3,English,1354,0.511,0.504,3
2,Chinese,103,0.408,0.273,3
8,Russian,94,0.479,0.481,3
7,Hindi,86,0.500,0.489,3
9,Spanish,83,0.494,0.482,3
0,Arabic,79,0.506,0.509,3
6,Greek,77,0.532,0.531,3
11,Thai,77,0.390,0.368,3
10,Swahili,72,0.347,0.329,3
1,Bulgarian,71,0.535,0.524,3


catboost · filas = clase real, columnas = predicción


,entailment,neutral,contradiction
entailment,476,175,184
neutral,204,406,166
contradiction,196,237,380


,precision,recall,f1-score,support
entailment,0.543,0.570,0.556,835.0
neutral,0.496,0.523,0.509,776.0
contradiction,0.521,0.467,0.493,813.0


,language,n,accuracy,macro_f1,n_classes
3,English,1354,0.575,0.571,3
2,Chinese,103,0.330,0.277,3
8,Russian,94,0.479,0.476,3
7,Hindi,86,0.453,0.447,3
9,Spanish,83,0.554,0.549,3
0,Arabic,79,0.342,0.335,3
6,Greek,77,0.468,0.461,3
11,Thai,77,0.377,0.370,3
10,Swahili,72,0.417,0.417,3
1,Bulgarian,71,0.563,0.563,3


## Pares para discutir en la defensa

Un acierto puede depender de una sola palabra que cambie el sentido del par. Recuperamos tres errores breves de CatBoost sobre los textos originales para poder leer ambos lados. Esta selección ilustra casos individuales. Para cuantificar un patrón hay que contar los errores de ese tipo en una muestra definida.

In [5]:
errors = val.assign(predicción=models["catboost"][2])
errors = errors.loc[(errors.label != errors.predicción) & errors.lang_abv.isin(["en", "es"])].copy()
errors["longitud"] = errors.premise.str.len() + errors.hypothesis.str.len()
errors["real"] = errors.label.map(LABEL_NAMES)
errors["predicha"] = errors.predicción.map(LABEL_NAMES)
errors.nsmallest(3, "longitud")[["id", "premise", "hypothesis", "real", "predicha"]]

,id,premise,hypothesis,real,predicha
1667,87fd77e7f2,Per week?,Every day.,contradiction,entailment
2146,4dcf1b9bb3,The family.,The couple.,neutral,entailment
1013,976407c049,was it bad,Was it not good?,entailment,contradiction


## Traducción en folds agrupados y validación

Volvemos a comparar ambos clasificadores con y sin traducción. Los tres folds usan la misma semilla, están agrupados por la premisa original y mantienen los mismos IDs. El holdout es el mismo para las cuatro ejecuciones. Las métricas de holdout se usan para describir rendimiento, no para reajustar parámetros.


In [6]:
import json
from sklearn.model_selection import StratifiedGroupKFold

translation_runs = json.loads((PROJECT / "results_translation_cv/results.json").read_text(encoding="utf-8"))["experiments"]
by_run = {(r["dataset"], r["classifier"]): r for r in translation_runs}
translation_summary = []
for model_name in ("naive_bayes", "catboost"):
    original = by_run[("original", model_name)]
    translated = by_run[("translated", model_name)]
    if model_name == "catboost":
        old_holdout = json.loads((PROJECT / "results_ollama_catboost/metrics.json").read_text(encoding="utf-8"))["arms"]
        original_holdout = old_holdout["original"]
        translated_holdout = old_holdout["translated"]
    else:
        original_holdout = original["holdout"]
        translated_holdout = translated["holdout"]
    translation_summary.append({
        "modelo": model_name,
        "accuracy CV original": original["summary"]["accuracy_mean"],
        "accuracy CV traducido": translated["summary"]["accuracy_mean"],
        "delta CV": translated["summary"]["accuracy_mean"] - original["summary"]["accuracy_mean"],
        "macro-F1 CV original": original["summary"]["macro_f1_mean"],
        "macro-F1 CV traducido": translated["summary"]["macro_f1_mean"],
        "accuracy holdout original": original_holdout["accuracy"],
        "accuracy holdout traducido": translated_holdout["accuracy"],
        "macro-F1 holdout original": original_holdout["macro_f1"],
        "macro-F1 holdout traducido": translated_holdout["macro_f1"],
    })
display(pd.DataFrame(translation_summary).set_index("modelo").round(4))

fold_comparison = []
for model_name in ("naive_bayes", "catboost"):
    original = by_run[("original", model_name)]["folds"]
    translated = by_run[("translated", model_name)]["folds"]
    for a, b in zip(original, translated):
        fold_comparison.append({
            "modelo": model_name, "fold": a["fold"],
            "accuracy original": a["accuracy"], "accuracy traducido": b["accuracy"],
            "macro-F1 original": a["macro_f1"], "macro-F1 traducido": b["macro_f1"],
        })
display(pd.DataFrame(fold_comparison).round(4))

,accuracy CV original,accuracy CV traducido,delta CV,macro-F1 CV original,macro-F1 CV traducido,accuracy holdout original,accuracy holdout traducido,macro-F1 holdout original,macro-F1 holdout traducido
modelo,,,,,,,,,
naive_bayes,0.4850,0.5026,0.0175,0.4828,0.4989,0.4847,0.4930,0.4828,0.4900
catboost,0.5219,0.5482,0.0263,0.5208,0.5482,0.5206,0.5656,0.5195,0.5655


,modelo,fold,accuracy original,accuracy traducido,macro-F1 original,macro-F1 traducido
0,naive_bayes,1,0.4938,0.4994,0.4919,0.4950
1,naive_bayes,2,0.4718,0.4966,0.4695,0.4937
2,naive_bayes,3,0.4895,0.5118,0.4868,0.5078
3,catboost,1,0.5198,0.5424,0.5198,0.5424
4,catboost,2,0.5192,0.5523,0.5182,0.5525
5,catboost,3,0.5266,0.5498,0.5244,0.5497


La auditoría confirma 9696 IDs únicos en desarrollo, 2424 en holdout y la misma etiqueta, idioma de origen y premisa original para cada ID en las dos condiciones. La partición de desarrollo es StratifiedGroupKFold, tres folds y semilla 42. No hay premisas originales compartidas entre entrenamiento y validación dentro de ningún fold. El holdout histórico sí comparte premisas con desarrollo, por lo que sus resultados no representan una prueba independiente de premisas nuevas.


In [7]:
train_original, val_original = train, val
if translations is None:
    print("CSV traducido ausente. Se conservan las tablas históricas y solo se comprueban los folds del original.")
else:
    for left in (train_original, val_original):
        right = translations.loc[left.id].reset_index()
        pd.testing.assert_frame_equal(left.reset_index(drop=True), right[left.columns])
    print("Comprobada la correspondencia de filas entre original y traducido.")
fold_splits = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42).split(
    train_original, train_original.label, groups=train_original.premise))
assert [len(valid_idx) for _, valid_idx in fold_splits] == [
    row["n_test"] for row in by_run[("original", "catboost")]["folds"]]
for fit_idx, valid_idx in fold_splits:
    assert set(train_original.premise.iloc[fit_idx]).isdisjoint(
        set(train_original.premise.iloc[valid_idx]))
print(len(train_original), "IDs de desarrollo y", len(val_original), "IDs de holdout.")

Auditoría correcta. 9696 IDs de desarrollo y 2424 IDs de holdout.
Los dos brazos comparten filas y cada fold mantiene las premisas originales fuera de su partición contraria.


## Qué cambió en ejemplos concretos

La traducción ayuda y también puede cambiar la predicción en una dirección incorrecta. Aquí se muestran pares no ingleses de la validación. La etiqueta real y las dos predicciones se recuperan por ID, junto con la traducción guardada.


In [8]:
paired = read_csv("results_ollama_catboost/predictions.csv").merge(
    val[["id", "premise", "hypothesis"]], on="id", validate="one_to_one")
text_columns = ["lang_abv", "premise", "hypothesis"]
if translations is not None:
    for column in ("premise_translated", "hypothesis_translated"):
        paired[column] = paired.id.map(translations[column])
        text_columns.append(column)
else:
    print("Textos traducidos omitidos: el CSV de traducciones no está disponible.")
paired["etiqueta real"] = paired.label.map(LABEL_NAMES)
paired["predicción original"] = paired.original.map(LABEL_NAMES)
paired["predicción traducida"] = paired.translated.map(LABEL_NAMES)
corrected_pair = paired.loc[paired.id.eq("3931fbe82a")]
regressed_pair = paired.loc[paired.id.eq("b962358e67")]
assert len(corrected_pair) == len(regressed_pair) == 1
for pair in (corrected_pair, regressed_pair):
    display(pair[text_columns + ["etiqueta real", "predicción original", "predicción traducida"]])

,lang_abv,premise,hypothesis,premise_translated,hypothesis_translated,etiqueta real,predicción original,predicción traducida
17,fr,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,Small things like these make a huge difference...,I was trying to accomplish something.,entailment,contradiction,entailment


,lang_abv,premise,hypothesis,premise_translated,hypothesis_translated,etiqueta real,predicción original,predicción traducida
10,vi,"Ông Nields trả lời rằng, tôi hoàn toàn hạnh ph...",Ông Nields nói ông rất vui khi sử dụng những t...,"Mr. Nields replied that, I am completely happy...",Mr. Nields said he was very happy to use those...,entailment,entailment,contradiction
